# 00 Sample design template

The one place to start. This notebook runs the whole sample design for TRPA's forest health plot network, top to bottom, from the sources the threshold analysis used: `F:\GIS\DB_CONNECT\Vector.sde` (read only), `Raster.sde`, and the threshold project geodatabase. Every number comes from `config.yaml`; every method comes from `src/`. Sections 0 to 7 each open with what the section does, what is already decided, what Andy decides here, and which module does the work.

**What the network is for.** Two adopted standards say what share of the Basin's 115,396 acres of conifer forest should look a certain way: VP9, seral stage and canopy cover, 75 percent of each forest type within its desired ranges; VP10, stand density, 50 percent within the TPA and basal area maxima. Today's figures come from a model. The plots measure the forest the way the standards are written, at enough places to make the 2028 evaluation defensible, and train the next imputation.

**Settled (do not reopen here).** Three forest types per report Table 1 (Sierran mixed conifer with lodgepole and white fir, red fir, Jeffrey pine with eastside pine and juniper). Frame excludes urban land use, designated wilderness, and non conifer types. The sampling unit is a 3 by 3 block of 30 m pixels with the plot on the centre pixel and a 120 m minimum separation. The plot itself (nested quarter acre primary, 56.4 m macroplot) is protocol, not design. Split sample: Half A equal probability within forest type, balanced on TPI class, basin side, and elevation band through `caty_n`; Half B unequal probability toward structural cells and rare types. 300 is the design target; 60, 100, and 300 are prefixes of one GRTS order. The frozen selection is `spsurvey::grts()` through `scripts/grts_split_draw.R`; the Python GRTS in `src/strata.py` is for iteration. Legacy sites are the Lake Tahoe West LiDAR validation plots and Hugh Safford's burn plots, nothing else; TEON sites never enter. Strata inputs: LiDAR 2022 metrics if they exist by Oct 1, 2026, otherwise the threshold rasters on F:, chosen by `strata.source`. Freeze Oct 16, 2026; RFP posts Nov 2.

**Synthetic mode.** With `run.synthetic: true` a toy Basin stands in for every source and the same section code runs without arcpy, F:, or R. Its numbers mean nothing beyond proving the chain. Flip it to `false` on the arcgispro-py3 interpreter with F: mounted.

## 0. Setup

Loads `config.yaml`, opens a log under `logs/`, and, on a real run, imports arcpy and points its workspace at a scratch file geodatabase under `data/processed/`. Nothing is ever written through an `.sde` connection. `reader` is the one door to every source: it reads the real layers through `src/layers.py` (arcpy for `.sde` and `.gdb`, rasterio for files, requests for REST) or hands back the toy Basin.

Decided: paths and the read only rule. Andy decides: nothing here. Module: `src/io.py`, `src/design_template.SourceReader`.

In [ ]:
import sys, os, time, json, shutil, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="Mean of empty slice")
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd, geopandas as gpd
from src.io import load_config, get_logger
from src import strata, qa, covariates, sample_size, tessellation as T
from src import design_template as dt
from src.layers import resolve_source, arcpy_available, is_arcgis_path

cfg = load_config()
log = get_logger("00_sample_design_template")
SYN, FREEZE = bool(cfg["run"]["synthetic"]), bool(cfg["run"]["freeze"])
WCRS = cfg["crs"]["working"]
AC = dt.M2_PER_ACRE
STAMP = pd.Timestamp.today().strftime("%Y%m%d")
TYPES = list(cfg["forest_types"]["population_acres"])
P = (Path.cwd() / cfg["paths"]["processed"]).resolve(); P.mkdir(parents=True, exist_ok=True)
O = (Path.cwd() / cfg["paths"]["outputs"]).resolve(); O.mkdir(exist_ok=True)
log.info(f"Project: {cfg['project']['name']} | synthetic={SYN} | freeze={FREEZE} | strata.source={cfg['strata']['source']}")

WORK = None
if not SYN:
    _t = time.time()
    import arcpy                                          # well over a minute in arcgispro-py3
    arcpy.CheckOutExtension("Spatial")
    WORK = (P / "design_template_work.gdb").as_posix()    # scratch only; nothing is ever written through an .sde connection
    if not arcpy.Exists(WORK):
        arcpy.management.CreateFileGDB(str(P), Path(WORK).name)
    arcpy.env.workspace = WORK
    arcpy.env.overwriteOutput = True
    log.info(f"arcpy {arcpy.GetInstallInfo()['Version']} imported in {time.time() - _t:.0f} s; workspace {WORK}")
else:
    arcpy = None
    log.info("synthetic run: arcpy not imported, F: not touched; the toy Basin stands in for every source")

reader = dt.SourceReader(cfg, log=log, synthetic=SYN, scratch_gdb=WORK)

## 1. Sources

Two things happen here. First, a discovery walk over `Vector.sde` (and the project gdb) lists every feature dataset and class whose name matches `sde.discovery_keywords`, so the `# CONFIRM` placeholders in `config.yaml` can be replaced with real names. Second, a status table of every entry under `sources:` says which form is picked (`rrk`, `lidar`, `sde`, or `file`) and whether it is found. Read the table before going further: a missing frame or strata source stops section 2.

Decided: the sources are SDE and the project gdb, not REST. Andy decides: the real class names for roads, trails, streams, structures, parcels, ownership, state line, treatments, and fire perimeters, pasted into `config.yaml`. Module: `src/design_template.walk_sde`, `source_status`; `src/layers.resolve_source`.

In [ ]:
if not SYN:
    found = dt.walk_sde(cfg["sde"]["vector"], cfg["sde"]["discovery_keywords"])
    print(f"Vector.sde: {len(found)} datasets and classes match the keyword list")
    with pd.option_context("display.max_rows", 500, "display.width", 200):
        print(found.to_string(index=False))
    found.to_csv(O / f"sde_discovery_{STAMP}.csv", index=False)
    with arcpy.EnvManager(workspace=cfg["sde"]["project_gdb"]):
        gdb_rasters = arcpy.ListRasters() or []
    print(f"\nproject gdb rasters ({len(gdb_rasters)}): {', '.join(sorted(gdb_rasters))}")
else:
    print("synthetic run: the discovery walk needs arcpy and Vector.sde; skipped")

In [ ]:
status = dt.source_status(cfg)
with pd.option_context("display.max_rows", 200, "display.width", 250, "display.max_colwidth", 110):
    print(status.to_string(index=False))
status.to_csv(O / f"source_status_{STAMP}.csv", index=False)
needed = ["veg_type", "boundary", "urban_boundary", "wilderness", "dem", "roads", "trails", "streams", "structures", "parcels",
          "rrk_seral", "rrk_cover", "rrk_tpa", "rrk_ba", "rrk_density_assessment"]
if cfg["strata"]["source"] == "lidar":
    needed += ["p95_height_30m", "canopy_cover_30m", "stem_density_30m"]
missing = [k for k in needed if not reader.available(k)]
if missing and not SYN:
    raise SystemExit(f"sources missing, fix config.yaml sources: {missing}")
log.info(f"required sources present: {len(needed) - len(missing)} of {len(needed)}" + (f"; missing {missing}" if missing else ""))

## 2. Frame

The population is the threshold report's assessment frame, rebuilt from the same rasters: `veg_type_nonurban` (RRK 2023 CWHR type, already clipped to non urban land use, 30 m California Teale Albers) mapped to the three TRPA types through its attribute table and `tessellation.whr_to_type`, inside the TRPA boundary, minus the urban land use classes (Mixed-Use, Residential, Tourist), minus Wilderness. Acres by type are checked against report Table 1 (70,909 / 23,037 / 21,450, total 115,396) and the notebook stops if the population is more than `frame.population_tolerance_pct` off. Then the design exclusions with area accounting: slope above `frame.max_slope_pct` from the DEM, and `frame.edge_buffer_m` around roads, trails, streams, structures, and parcel edges. Then the sampling units: 3 by 3 blocks of pixels anchored at the raster origin, kept when the centre pixel is in the population and at least `frame.min_unit_fraction` of the block is. `acres` is the population area a unit represents, `unit_acres` the block area.

**The Wilderness question.** The report says the 115,396 acre frame excludes designated wilderness. The threshold notebook's non urban clause (`Description IN ('Wilderness', 'Resort Recreation', 'Recreation', 'Conservation', 'Backcountry')`) KEEPS the Wilderness class, and the crosswalk read from `veg_type_nonurban` totals 115,600 acres with it kept. Both readings are printed below against 115,396; `frame.exclude_wilderness` applies the one Andy confirms.

**The grid question.** The units are defined on the grid the strata come from. Today that is the 30 m Albers grid of the F: rasters; the LiDAR products will be 30 m EPSG:26910. Swapping `strata.source` re-defines every unit, so nothing selected on one grid carries to the other. Unit centres are carried in both the raster CRS (`x_src`, `y_src`) and the working CRS (`x`, `y`, EPSG:26910), where the selection, the hex keys, and the exports live. The DEM is projected and snapped to the strata grid before slope, aspect, and TPI are computed, so every attribute sits on one grid.

Decided: population definition, unit size, minimum fraction, buffers, slope cutoff. Andy decides: the Wilderness answer, and whether the buffer set and slope cutoff stay as configured. Module: `src/design_template` (frame helpers), `src/strata.block_reduce`, `src/strata.access_class`.

In [ ]:
veg, TR, RCRS = reader.raster("veg_type")
rat = reader.rat("veg_type")
SHAPE = veg.shape
CELL = float(abs(TR.a))
W2T = cfg["tessellation"]["whr_to_type"]
type_code = dt.forest_type_array(veg, rat, W2T)
whr = dt.whrtype_array(veg, rat)
log.info(f"frame raster {SHAPE}, {CELL:.0f} m, CRS {RCRS.to_string()}; working CRS {WCRS}")

boundary = reader.layer("boundary")
urban = reader.layer("urban_boundary")
wilderness = reader.layer("wilderness")
in_bdy = dt.rasterize_mask(boundary, TR, SHAPE, RCRS)
m_urban = dt.rasterize_mask(urban, TR, SHAPE, RCRS)
m_wild = dt.rasterize_mask(wilderness, TR, SHAPE, RCRS)
conifer = (type_code > 0) & in_bdy & ~m_urban

def acres_by_type(mask):
    return {t: float((mask & (type_code == c)).sum()) * CELL * CELL / AC for t, c in dt.TYPE_CODE.items()}

report = cfg["forest_types"]["population_acres"]
tol = cfg["frame"]["population_tolerance_pct"]
kept, ok_kept = dt.population_check(acres_by_type(conifer), report, tol)
dropped, ok_dropped = dt.population_check(acres_by_type(conifer & ~m_wild), report, tol)
print("Wilderness KEPT (threshold notebook clause):"); print(kept.to_string(index=False))
print("\nWilderness REMOVED (report text):"); print(dropped.to_string(index=False))
print(f"\nwilderness overlaps {float((conifer & m_wild).sum()) * CELL * CELL / AC:,.0f} acres of conifer frame")

EXCL_WILD = bool(cfg["frame"]["exclude_wilderness"])
population = conifer & ~m_wild if EXCL_WILD else conifer
pop_tab, pop_ok = (dropped, ok_dropped) if EXCL_WILD else (kept, ok_kept)
log.info(f"frame.exclude_wilderness={EXCL_WILD}: population {pop_tab.iloc[-1]['acres']:,.0f} ac vs report {sum(report.values()):,}")
if not pop_ok:
    raise SystemExit(f"population is {pop_tab.iloc[-1]['diff_pct']}% from report Table 1 (tolerance {tol}%). "
                     "Check the Wilderness setting, the urban clause, and whr_to_type before going on.")

In [ ]:
# terrain on the strata grid: the 0.7 m SDE bare earth is projected and snapped to the veg raster on a real run
dem, dem_tr, _ = reader.raster_on_grid("dem", template_key="veg_type")
assert dem.shape == SHAPE, f"DEM grid {dem.shape} does not match the frame raster {SHAPE}; check raster_on_grid"
slope_pct, aspect_deg = dt.terrain(dem, CELL)
steep = population & (np.nan_to_num(slope_pct, nan=0) > cfg["frame"]["max_slope_pct"])

# edge buffers: roads, trails, streams, structures, and parcel edges
buf = cfg["frame"]["edge_buffer_m"]
edge_layers = {k: reader.layer(k) for k in ["roads", "trails", "streams", "structures"] if reader.available(k)}
parcels = reader.layer("parcels") if reader.available("parcels") else None
m_edge = dt.rasterize_mask(list(edge_layers.values()), TR, SHAPE, RCRS, buffer_m=buf)
if parcels is not None:
    m_edge |= dt.rasterize_mask(parcels, TR, SHAPE, RCRS, buffer_m=buf, boundary_only=True)
for k, v in edge_layers.items():
    log.info(f"{k}: {len(v):,} features in the buffer set")
if parcels is None:
    log.warning("parcels not available; parcel edges are not buffered")

# distance to a road for the access class, on the grid
from scipy.ndimage import distance_transform_edt
m_road = dt.rasterize_mask(edge_layers.get("roads"), TR, SHAPE, RCRS)
dist_road = distance_transform_edt(~m_road) * CELL if m_road.any() else np.full(SHAPE, np.nan)

acct = [{"step": "population (report frame)", "acres": float(population.sum()) * CELL * CELL / AC}]
frame_px = population & ~steep
acct.append({"step": f"removed slope > {cfg['frame']['max_slope_pct']}%", "acres": float(steep.sum()) * CELL * CELL / AC})
edge_px = frame_px & m_edge
acct.append({"step": f"removed within {buf} m of a road, trail, stream, structure, or parcel edge", "acres": float(edge_px.sum()) * CELL * CELL / AC})
frame_px = frame_px & ~m_edge
acct.append({"step": "frame pixels", "acres": float(frame_px.sum()) * CELL * CELL / AC})

In [ ]:
# sampling units: k x k blocks anchored at the raster origin, plot on the centre pixel
k = int(cfg["frame"]["unit_px"])
units, bi, bj = dt.build_units(population, type_code, k, cfg["frame"]["min_unit_fraction"], TR)
units["whr_type"] = [whr[r * k + k // 2, c * k + k // 2] for r, c in zip(units["unit_row"], units["unit_col"])]
units["elev_m"] = dt.sample_units(dem, population, k, bi, bj, "mean")
units["slope_pct"] = dt.sample_units(slope_pct, population, k, bi, bj, "center")
units["aspect_deg"] = dt.sample_units(aspect_deg, population, k, bi, bj, "center")
units["dist_road_m"] = dt.sample_units(dist_road, population, k, bi, bj, "center")
units["near_edge"] = dt.sample_units(m_edge.astype(float), population, k, bi, bj, "center") > 0
units["in_wilderness"] = dt.sample_units(m_wild.astype(float), population, k, bi, bj, "center") > 0

# RRK structure at every unit, always: strata in rrk mode, evaluation in both modes
for key, col, how in [("rrk_seral", "rrk_seral", "center"), ("rrk_cover", "rrk_cover_pct", "mean"),
                      ("rrk_tpa", "rrk_tpa", "mean"), ("rrk_ba", "rrk_ba", "mean"), ("rrk_density_assessment", "rrk_density_exceeds", "center")]:
    arr, tr, _ = reader.raster(key, form="rrk")
    assert arr.shape == SHAPE and abs(tr.a - TR.a) < 1e-6 and abs(tr.c - TR.c) < 1e-3 and abs(tr.f - TR.f) < 1e-3, \
        f"{key} is not on the frame raster grid ({arr.shape} vs {SHAPE}); snap it to veg_type_nonurban first"
    units[col] = dt.sample_units(arr, population, k, bi, bj, how)
if cfg["strata"]["source"] == "lidar":
    for key, col in [("p95_height_30m", "p95_height_m"), ("stem_density_30m", "stem_density"), ("canopy_cover_30m", "canopy_cover_pct")]:
        arr, tr, _ = reader.raster(key, form="lidar")
        assert arr.shape == SHAPE and abs(tr.a - TR.a) < 1e-6, f"{key} is not on the frame grid; lidar mode needs veg_type on the LiDAR grid too"
        units[col] = dt.sample_units(arr, population, k, bi, bj, "mean")

# polygon attributes at the centre pixel: state, owner, management zone, disturbance flags
def category_at_centre(key, field_default):
    if not reader.available(key):
        return np.array([None] * len(units), dtype=object)
    lyr = reader.layer(key)
    field = reader.option(key, "field", field_default)
    if field not in lyr:
        log.warning(f"{key}: field {field} not found (has {list(lyr.columns)[:8]}); set sources.{key}.field")
        return np.array([None] * len(units), dtype=object)
    arr, lut = dt.rasterize_category(lyr, field, TR, SHAPE, RCRS)
    v = dt.sample_units(arr.astype(float), population, k, bi, bj, "center")
    return np.array([lut.get(int(c)) if c > 0 else None for c in np.nan_to_num(v)], dtype=object)
units["state"] = category_at_centre("state_line", cfg["tessellation"]["state_field"])
units["owner"] = category_at_centre("ownership", "OWNER")
units["mgmt_zone"] = category_at_centre("mgmt_zone", "ZONE_NAME")
for key, col in [("fire_severity", "post_fire"), ("treatments_2027_2031", "treatment_2027_2031")]:
    m = dt.rasterize_mask(reader.layer(key), TR, SHAPE, RCRS) if reader.available(key) else np.zeros(SHAPE, bool)
    units[col] = dt.sample_units(m.astype(float), population, k, bi, bj, "center") > 0

units = dt.to_working_crs(units, RCRS, WCRS)
log.info(f"units on the {CELL:.0f} m grid: {len(units):,} blocks of {k}x{k}; centres carried in {RCRS.to_string()} and {WCRS}")

In [ ]:
# unit level exclusions with accounting, then the access class
n0 = len(units)
acct.append({"step": "candidate units", "acres": units["acres"].sum(), "units": n0})
steep_u = units["slope_pct"] > cfg["frame"]["max_slope_pct"]
acct.append({"step": f"units removed, centre slope > {cfg['frame']['max_slope_pct']}%", "acres": units.loc[steep_u, "acres"].sum(), "units": int(steep_u.sum())})
units = units[~steep_u]
edge_u = units["near_edge"]
acct.append({"step": f"units removed, centre within {buf} m of an edge", "acres": units.loc[edge_u, "acres"].sum(), "units": int(edge_u.sum())})
units = units[~edge_u]
if cfg["frame"]["max_access_distance_m"]:
    far = units["dist_road_m"] > cfg["frame"]["max_access_distance_m"]
    acct.append({"step": "units removed beyond access distance", "acres": units.loc[far, "acres"].sum(), "units": int(far.sum())})
    units = units[~far]
if cfg["strata"]["source"] == "lidar" and "solid_frac" in units and cfg["frame"].get("max_solid_fraction"):
    solid = (units["solid_frac"] > cfg["frame"]["max_solid_fraction"]) & (units["canopy_cover_pct"] > cfg["threshold"]["cover_sparse_pct"])
    acct.append({"step": "units removed, solid surface", "acres": units.loc[solid, "acres"].sum(), "units": int(solid.sum())})
    units = units[~solid]
units = units.copy()
units["access_class"] = strata.access_class(units["dist_road_m"].fillna(1e9), units["slope_pct"].fillna(0), cfg)
acct.append({"step": "frame units", "acres": units["acres"].sum(), "units": len(units)})
acct = pd.DataFrame(acct)
acct.to_csv(O / "frame_accounting.csv", index=False)
print(acct.round(0).to_string(index=False))
print("\nframe acres by type (units):"); print(units.groupby("forest_type")["acres"].sum().round().to_string())
print("\naccess class:"); print(units["access_class"].value_counts().sort_index().to_string())
units.to_parquet(P / "frame.parquet", index=False)

## 3. Strata and covariates

Forest type is the fixed stratum. The structure axes come from the raster set `strata.source` names. In `rrk` mode they are already at the threshold decision points: seral 1, 2, 3 straight from `seral_stage_nonurban`; open or closed from `canopy_cover_nonurban` at 40 percent for Jeffrey pine and 50 for the others; density from `stand_density_threshold_assessment`, 0 at or below the TPA and basal area maxima and 1 above. In `lidar` mode they are p95 height at `strata.height_breaks_m`, stem density at `strata.density_breaks`, and cover from the LiDAR cover raster. Either way the classes are assigned by `strata.classify_cells` and cells under `strata.min_cell_acres` are collapsed within type by `strata.collapse_small_cells`; `design_template.strata_config_for_source` only tells `classify_cells` where the breaks sit for the chosen source. Cover is an attribute unless `strata.use_cover_axis` is on.

Balance covariates for Half A, from `src/covariates.py`: TPI class from the DEM on the strata grid (`split_sample.balance.tpi_radius_px`), basin side east or west of the lake's long axis, and elevation band at `split_sample.balance.elev_bands_m`. They are not strata; `caty_n` sets the expected count per category within each forest type so the equal probability half covers them.

Decided: the axes, the threshold decision points, forest type as the fixed stratum. Andy decides: the Oct 1 source call with Mason, the height and density breaks (lidar mode) and the cell collapse floor with Becky, the TPI radius, the lake axis anchor, and the elevation band edges. Module: `src/strata.classify_cells`, `collapse_small_cells`; `src/covariates`; `src/design_template.strata_config_for_source`, `proxy_columns`, `elevation_band`.

In [ ]:
frame = pd.read_parquet(P / "frame.parquet")
cfg_s = dt.strata_config_for_source(cfg)
frame = dt.proxy_columns(frame, cfg)
classified = strata.classify_cells(frame, cfg_s)
cells_raw = classified.groupby(["cell_id", "forest_type", "seral_class", "density_class"], as_index=False)["acres"].sum()
log.info(f"strata.source={cfg['strata']['source']}: {len(cells_raw)} raw cells; below {cfg['strata']['min_cell_acres']} ac: {int((cells_raw['acres'] < cfg['strata']['min_cell_acres']).sum())}")
classified, collapse_log = strata.collapse_small_cells(classified, cfg, log)
collapse_log.to_csv(O / "cell_collapse_log.csv", index=False)
cells = classified.groupby(["cell_id", "forest_type", "seral_class", "density_class"], as_index=False)["acres"].sum().sort_values("cell_id")
cells["share_of_type"] = cells["acres"] / cells.groupby("forest_type")["acres"].transform("sum")
cells.to_csv(O / "cells.csv", index=False)
print(cells.round(3).to_string(index=False))
if len(collapse_log):
    print("\ncollapsed:"); print(collapse_log.to_string(index=False))
xt = pd.crosstab(classified["cell_id"], classified["cover_class"], values=classified["acres"], aggfunc="sum").fillna(0).round()
xt.to_csv(O / "cells_by_cover.csv")

In [ ]:
# balance covariates: TPI class, basin side, elevation band
bal = cfg["split_sample"]["balance"]
tpi_grid = covariates.tpi(np.where(np.isnan(dem), np.nanmean(dem), dem), int(bal["tpi_radius_px"]))
classified["tpi"] = [tpi_grid[r * k + k // 2, c * k + k // 2] for r, c in zip(classified["unit_row"], classified["unit_col"])]
classified["tpi_class"] = covariates.tpi_class(classified["tpi"].to_numpy(), classified["slope_pct"].fillna(0).to_numpy())
anchor = bal["lake_axis_anchor_xy"]
if anchor is None:
    c = boundary.to_crs(WCRS).union_all().centroid
    anchor = [float(c.x), float(c.y)]
    log.info(f"lake_axis_anchor_xy is null; using the boundary centroid {anchor[0]:.0f}, {anchor[1]:.0f}")
classified["basin_side"] = covariates.basin_side(classified["x"].to_numpy(), classified["y"].to_numpy(), anchor[0], anchor[1], bal["lake_axis_azimuth_deg"])
classified["elev_band"] = dt.elevation_band(classified["elev_m"].fillna(classified["elev_m"].median()), bal["elev_bands_m"])
classified["balance_caty"] = covariates.balance_category(classified["basin_side"].to_numpy(), classified["tpi_class"].to_numpy())
classified["balance_caty"] = classified["balance_caty"] + "_" + classified["elev_band"]
classified.to_parquet(P / "frame_classified.parquet", index=False)
print(pd.crosstab(classified["forest_type"], classified["balance_caty"], values=classified["acres"], aggfunc="sum").fillna(0).round().T.to_string())

## 4. Allocation

Half A gets `split_sample.half_a_share` of each nested level, split across forest types by `split_sample.half_a_by_type` (area shares), with `caty_n` per type from `covariates.caty_n_by_stratum`. Half B gets the rest, allocated across the structural cells by `strata.allocate` (Candidate B: floor per cell, type shares, seral by density shares, tail boost) with the Nevada floor reported by `strata.apply_nevada_floor`. Both are printed at 60, 100, and 300 so the nested prefixes are visible. When the per cell floor times the cell count exceeds Half B's share of a level, `design_template.allocate_half_b` lowers the floor for that level and says so; that is a decision, not a fix. `sample_size.detectable_margin` shows what each type's count buys: the smallest margin above the 75 percent (VP9) and 50 percent (VP10) standards that many plots can detect at alpha 0.05 and power 0.80.

Decided: the split, the levels, 300 as the design target. Andy decides: Half B weights (type shares, seral and density shares, tail boost, floor per cell) with Becky, and whether the min level floor holds. Module: `src/strata.allocate`, `apply_nevada_floor`; `src/covariates.caty_n_by_stratum`; `src/sample_size.detectable_margin`; `src/design_template.half_a_counts`, `allocate_half_b`.

In [ ]:
LEVELS = list(cfg["allocation"]["levels"])
nA_by_level = {lvl: dt.half_a_counts(cfg, lvl) for lvl in LEVELS}
allocA = pd.DataFrame([{"level": lvl, "forest_type": t, "n_A": n} for lvl, d in nA_by_level.items() for t, n in d.items()])
catyA = {}
for lvl in LEVELS:
    cn = covariates.caty_n_by_stratum(classified, nA_by_level[lvl], stratum_col="forest_type", caty_col="balance_caty", min_per_caty=1)
    catyA[lvl] = pd.DataFrame([{"level": lvl, "forest_type": t, "balance_caty": c, "n": n} for t, d in cn.items() for c, n in d.items()])
    over = {t: sum(d.values()) - nA_by_level[lvl][t] for t, d in cn.items() if sum(d.values()) != nA_by_level[lvl][t]}
    if over:
        log.warning(f"{lvl}: caty_n cannot honour every balance category at this level (over by {over}); only the full level's caty_n goes to spsurvey, the prefixes inherit its balance")
allocB = pd.concat([dt.allocate_half_b(cells, cfg, lvl, log) for lvl in LEVELS], ignore_index=True)
allocB = strata.apply_nevada_floor(allocB, classified.groupby(["cell_id", "state"], as_index=False)["acres"].sum(), cfg)
allocB.to_csv(O / "allocation_table.csv", index=False)
allocA.to_csv(O / "allocation_half_a.csv", index=False)
pd.concat(catyA.values()).to_csv(O / "caty_n_half_a.csv", index=False)

print("Half A per type (equal probability, spatially balanced):")
print(allocA.pivot(index="forest_type", columns="level", values="n_A")[LEVELS].to_string())
print("\nHalf B per cell (n_B, unequal probability; n_A shown for the proportional candidate):")
print(allocB.pivot_table(index=["forest_type", "seral_class", "density_class"], columns="level", values=["n_B", "n_A"]).astype(int)
      .reindex(columns=pd.MultiIndex.from_product([["n_B", "n_A"], LEVELS])).to_string())
for lvl in LEVELS:
    a = allocB[allocB["level"] == lvl]
    nv = a.groupby("forest_type")["expected_nv_plots"].sum().round(1).to_dict()
    print(f"\n{lvl}: total {a['n_B'].sum() + sum(nA_by_level[lvl].values())} = A {sum(nA_by_level[lvl].values())} + B {a['n_B'].sum()}; expected NV plots in B {nv}; NV floor ok {a.groupby('forest_type')['nv_floor_ok'].first().to_dict()}")
    if a["note"].iloc[0]:
        print("   note:", a["note"].iloc[0])

In [ ]:
rows = []
for lvl in LEVELS:
    for t in TYPES:
        n = nA_by_level[lvl][t] + int(allocB.loc[(allocB["level"] == lvl) & (allocB["forest_type"] == t), "n_B"].sum())
        rows.append({"level": lvl, "forest_type": t, "n_total": n, "n_A": nA_by_level[lvl][t], "n_B": n - nA_by_level[lvl][t],
                     "vp9_margin_pts": round(100 * sample_size.detectable_margin(n, 0.75), 1),
                     "vp10_margin_pts": round(100 * sample_size.detectable_margin(n, 0.50), 1),
                     "ci_halfwidth_p50_pts": round(100 * sample_size.ci_halfwidth(n, 0.5), 1)})
prec = pd.DataFrame(rows)
prec.to_csv(O / "precision_by_type.csv", index=False)
print("smallest margin above the standard each type's count can detect (alpha 0.05, power 0.80), and the 95% CI half width at p = 0.5:")
print(prec.to_string(index=False))

## 5. Selection

Writes the five inputs `scripts/grts_split_draw.R` reads (`frame_points.gpkg`, `allocation_typeA.csv`, `caty_n_typeA.csv`, `allocation_full.csv`, `legacy_sites.gpkg`) and calls it through `run.rscript` with the seed, the oversample factor, and the minimum distance from `draw:`. spsurvey is the frozen selection, for parity with TEON's backbone. When Rscript is not on the machine, or the call fails, the Python GRTS in `src/strata.py` makes the same split selection for iteration and the log says which one ran. Legacy sites (`draw.legacy_sources`, LTW and burn plots only) are attached to the unit that contains them, enter Half A as legacy sites, and count toward their type. `run.freeze` guards the seed: a frozen run refuses to proceed if the seed in config differs from the one recorded in `outputs/frozen_seed.json`, and records it the first time.

Decided: spsurvey for the frozen selection, seed 20261016, 120 m separation, oversample 2.5, legacy sources. Andy decides: nothing on the method; the backup list rule (section 7) is his. Module: `scripts/grts_split_draw.R`; `src/strata.grts_draw` (fallback); `src/design_template.write_selection_inputs`, `run_rscript`, `python_split_selection`, `normalise_spsurvey_sites`.

In [ ]:
frame = classified.copy()
# inclusion weights for Half B: disturbance targeting, representativeness when available
ds = cfg["allocation"]["disturbance_shares"]
w = pd.Series(1.0, index=frame.index)
for flag, share in [("post_fire", ds["post_fire"]), ("treatment_2027_2031", ds["treatment"])]:
    p_frame = float(frame[flag].mean())
    if 0 < p_frame < share:
        w[frame[flag]] *= share / p_frame
    log.info(f"{flag}: frame share {p_frame:.3f}, target {share:.2f}")
if cfg["draw"]["weight_by_representativeness"] and "representativeness" in frame:
    w *= 1 + frame["representativeness"].rank(pct=True)
frame["inclusion_weight"] = w

# legacy sites: LTW LiDAR validation plots and Safford burn plots, attached to the unit that contains them
frame["legacy"], frame["legacy_source"] = False, None
half_diag = CELL * k * 0.7072
pts = dt.units_as_points(frame, WCRS)
for key in cfg["draw"]["legacy_sources"]:
    if not reader.available(key):
        log.warning(f"{key}: not available yet (partner file); no legacy sites from it"); continue
    leg = reader.layer(key)
    near = gpd.sjoin_nearest(leg[["geometry"]], pts[["unit_id", "geometry"]], max_distance=half_diag)
    hit = frame["unit_id"].isin(near["unit_id"])
    frame.loc[hit, "legacy"] = True
    frame.loc[hit & frame["legacy_source"].isna(), "legacy_source"] = key
    log.info(f"{key}: {len(leg)} plots, {near['unit_id'].nunique()} inside a frame unit")
# other units within the existing plot buffer step aside
if frame["legacy"].any():
    from scipy.spatial import cKDTree
    d, _ = cKDTree(frame.loc[frame["legacy"], ["x", "y"]].to_numpy(float)).query(frame[["x", "y"]].to_numpy(float))
    crowd = (d < cfg["frame"]["exclude_existing_plot_buffer_m"]) & ~frame["legacy"].to_numpy()
    log.info(f"{int(crowd.sum())} units within {cfg['frame']['exclude_existing_plot_buffer_m']} m of a legacy plot removed")
    frame = frame[~crowd].copy()
log.info(f"legacy sites in frame: {int(frame['legacy'].sum())}")

# seed guard
seed_file = O / "frozen_seed.json"
if FREEZE:
    if seed_file.exists():
        rec = json.loads(seed_file.read_text())
        if int(rec["seed"]) != int(cfg["draw"]["seed"]):
            raise SystemExit(f"run.freeze is true and draw.seed {cfg['draw']['seed']} differs from the frozen seed {rec['seed']} ({rec['date']}). Restart review or restore the seed.")
    else:
        seed_file.write_text(json.dumps({"seed": int(cfg["draw"]["seed"]), "date": STAMP, "strata_source": cfg["strata"]["source"]}, indent=2))
        log.info(f"FROZEN: seed {cfg['draw']['seed']} recorded in {seed_file.name}")

# inputs for spsurvey at the full level; the nested prefixes come from the order
full_A = pd.DataFrame([{"forest_type": t, "n_A": n} for t, n in nA_by_level["full"].items()])
full_B = allocB[allocB["level"] == "full"]
paths = dt.write_selection_inputs(frame, full_A, catyA["full"], full_B, cfg, P, log)

In [ ]:
res = None if SYN else dt.run_rscript(cfg, Path("scripts/grts_split_draw.R"), [cfg["draw"]["seed"], cfg["draw"]["oversample_factor"], cfg["draw"]["min_distance_m"]], Path.cwd(), log)
r_out = O / "grts_split_draw.gpkg"
if res is not None and res.returncode == 0 and r_out.exists():
    sel = dt.normalise_spsurvey_sites(gpd.read_file(r_out), frame)
    SELECTION = "spsurvey::grts (scripts/grts_split_draw.R)"
else:
    if SYN:
        log.info("synthetic run: the Python GRTS makes the split selection")
    else:
        log.warning("spsurvey selection not available; falling back to the Python GRTS. This is NOT the frozen selection, and caty_n is not applied (Half A is equal probability within type only).")
    sel = dt.python_split_selection(frame, nA_by_level["full"], full_B.set_index("cell_id")["n_B"].to_dict(), cfg, log)
    SELECTION = "python strata.grts_draw (iteration only)"
log.info(f"selection by {SELECTION}: {len(sel)} rows; " + ", ".join(f"{h} {s} {n}" for (h, s), n in sel.groupby(["half", "status"]).size().items()))
print(sel.groupby(["half", "status"]).size().unstack(fill_value=0).to_string())

In [ ]:
# nested levels and one installation order for both halves; blind remeasurement flag
sel = dt.install_order_split(sel, cfg, allocB, nA_by_level)
rng = np.random.default_rng(int(cfg["draw"]["seed"]) + 1)
prim_min = sel["status"].isin(["primary", "legacy"]) & (sel["level"] == "min")
sel["remeasure_flag"] = False
n_re = int(round(prim_min.sum() * cfg["draw"]["remeasure_share"]))
if n_re:
    sel.loc[rng.choice(sel.index[prim_min], n_re, replace=False), "remeasure_flag"] = True
sel["plot_id"] = "FH-" + sel["half"] + "-" + sel["stratum"].astype(str) + "-" + sel.groupby("stratum").cumcount().add(1).astype(str).str.zfill(3)
sel.to_parquet(P / "selected.parquet", index=False)
print(sel.groupby(["half", "level"]).size().unstack(fill_value=0).to_string())

## 6. Evaluate

Before anything freezes: spatial balance of each half (spsurvey's `sp_balance` when the R script ran, otherwise a quadrant chi square against the frame's own distribution, where 1 is what simple random sampling gives; legacy sites and Half B's unequal probabilities push it up by design, so read it half by half and against the Python fallback's own baseline), realized versus expected `caty_n` counts for Half A, per type counts by level, VP9 and VP10 class coverage of the primaries against the RRK rasters (every class of every type should be represented at the minimum level), access class counts, a with and without legacy comparison, and the QA report from `src/qa.py`. `sites_closer_than_min_distance` counts pairs under `draw.min_distance_m`; new sites never violate it, so any pair it finds is two legacy plots closer than 120 m, and which of the two stays is a decision for Andy and Becky. The VP9 and VP10 cross tabs read the `rrk` form of the evaluation sources whatever `strata.source` is, so in lidar mode they are an independent check of the strata and in rrk mode they restate them.

Decided: what is checked. Andy decides: what to do about a flag (collapse, reweight, or accept and record). Module: `src/qa.py`; `src/design_template.spatial_balance`, `vp9_class`, `vp10_class`, `coverage_table`.

In [ ]:
sel = pd.read_parquet(P / "selected.parquet")
frame = pd.read_parquet(P / "frame_classified.parquet")
prim = sel[sel["status"].isin(["primary", "legacy"])].copy()
new = prim[prim["status"] == "primary"]

# 1. spatial balance
bal_file = O / "grts_split_balance.csv"
if bal_file.exists() and SELECTION.startswith("spsurvey"):
    print("spsurvey sp_balance:"); print(pd.read_csv(bal_file).to_string(index=False))
sb = {h: dt.spatial_balance(prim.loc[prim["half"] == h, ["x", "y"]].to_numpy(float), frame[["x", "y"]].to_numpy(float)) for h in ("A", "B")}
sb["A+B"] = dt.spatial_balance(prim[["x", "y"]].to_numpy(float), frame[["x", "y"]].to_numpy(float))
print("quadrant chi-square per df (1 is simple random sampling; balanced sits below):", sb)

# 2. Half A realized vs expected caty counts
real = prim[prim["half"] == "A"].groupby(["forest_type", "balance_caty"]).size().rename("n_realized")
caty_chk = catyA["full"].set_index(["forest_type", "balance_caty"])["n"].rename("n_expected").to_frame().join(real).fillna(0).astype(int)
caty_chk["diff"] = caty_chk["n_realized"] - caty_chk["n_expected"]
caty_chk.to_csv(O / "eval_caty_check.csv")
print("\nHalf A categories off by more than 1" + (" (expected with the Python fallback, which does not apply caty_n)" if not SELECTION.startswith("spsurvey") else "") + ":")
print(caty_chk[caty_chk["diff"].abs() > 1].to_string())

# 3. per type counts by level
cov = prim.groupby(["forest_type", "half", "level"]).size().unstack("level", fill_value=0)
cov = cov.reindex(columns=[c for c in LEVELS + ["beyond"] if c in cov.columns])
for i, lvl in enumerate(LEVELS):
    cov[f"cum_{lvl}"] = cov[LEVELS[:i + 1]].sum(axis=1)
cov.to_csv(O / "eval_class_coverage.csv")
print("\nprimaries by type, half, and level (cum_* are the nested totals):"); print(cov.to_string())

In [ ]:
# 4. VP9 and VP10 class coverage against the RRK rasters
cut = cfg["threshold"]["cover_open_closed_pct"]
for df in (frame, prim):
    df["vp9_class"] = dt.vp9_class(df["rrk_seral"], df["rrk_cover_pct"], df["forest_type"], cut)
    df["vp10_class"] = dt.vp10_class(df["rrk_seral"], df["rrk_density_exceeds"])
at_min = prim[prim["level"] == "min"]
vp9 = dt.coverage_table(frame[frame["vp9_class"] != ""], prim[prim["vp9_class"] != ""], "vp9_class")
vp9["n_at_min"] = at_min.groupby(["forest_type", "vp9_class"]).size().reindex(vp9.index).fillna(0).astype(int)
vp10 = dt.coverage_table(frame[frame["vp10_class"] != ""], prim[prim["vp10_class"] != ""], "vp10_class")
vp10["n_at_min"] = at_min.groupby(["forest_type", "vp10_class"]).size().reindex(vp10.index).fillna(0).astype(int)
vp9.to_csv(O / "eval_vp9_crosstab.csv"); vp10.to_csv(O / "eval_vp10_crosstab.csv")
empty_vp9, empty_vp10 = int((vp9["n_at_min"] == 0).sum()), int((vp10["n_at_min"] == 0).sum())
print("VP9 classes (frame share vs primaries; n_at_min is the 60 plot prefix):"); print(vp9.to_string())
print("\nVP10 classes:"); print(vp10.to_string())
log.info(f"type x class combinations empty at the min level: VP9 {empty_vp9}, VP10 {empty_vp10}")

# 5. access class, owner, state
acc = prim.groupby(["level", "access_class"]).size().unstack(fill_value=0)
print("\naccess class by level:"); print(acc.to_string())
print("\nstate by half:"); print(pd.crosstab(prim["half"], prim["state"].fillna("?")).to_string())
pd.concat({"access": acc, "owner": prim.groupby(["level", "owner"]).size().unstack(fill_value=0),
           "state": prim.groupby(["level", "state"]).size().unstack(fill_value=0)}, axis=1).to_csv(O / "eval_practicality.csv")

# 6. with and without legacy
cmp = pd.DataFrame({"with_legacy": prim.groupby("forest_type").size(), "new_only": new.groupby("forest_type").size(),
                    "legacy": prim[prim["status"] == "legacy"].groupby("forest_type").size()}).fillna(0).astype(int)
cmp["legacy_share_pct"] = (100 * cmp["legacy"] / cmp["with_legacy"]).round(1)
print("\nlegacy contribution by type:"); print(cmp.to_string())
covs = [c for c in ["elev_m", "slope_pct", "aspect_deg", "rrk_cover_pct", "rrk_tpa", "dist_road_m"] if c in frame]
from scipy import stats
rows = []
for c in covs:
    for label, s in (("with_legacy", prim), ("new_only", new)):
        ks = stats.ks_2samp(s[c].dropna(), frame[c].dropna())
        rows.append({"covariate": c, "sample": label, "n": len(s), "ks_stat": round(ks.statistic, 3), "ks_p": round(ks.pvalue, 3),
                     "smd": round((s[c].mean() - frame[c].mean()) / frame[c].std(), 3)})
balance = pd.DataFrame(rows); balance.to_csv(O / "eval_covariate_balance.csv", index=False)
flag = balance[(balance["smd"].abs() > 0.25) & (balance["ks_p"] < 0.05)]
print("\ncovariate imbalance flags (|SMD| > 0.25 and KS p < 0.05):"); print(flag.to_string(index=False) if len(flag) else "none")

In [ ]:
checks = {
    "selection": {"method": SELECTION, "seed": int(cfg["draw"]["seed"]), "strata_source": cfg["strata"]["source"], "synthetic": SYN},
    "nulls": qa.check_nulls(prim, ["plot_id", "cell_id", "x", "y", "forest_type", "access_class", "level"]),
    "duplicate_plot_ids": qa.check_duplicates(sel, ["plot_id"]),
    "duplicate_units": qa.check_duplicates(sel, ["unit_id"]),
    "sites_closer_than_min_distance": qa.check_min_distance(prim, cfg["draw"]["min_distance_m"]),
    "row_count_primary": qa.check_row_count(prim, expected_min=cfg["allocation"]["levels"]["min"], expected_max=cfg["allocation"]["levels"]["full"] + 60),
    "forest_type_domain": qa.check_value_domain(prim, "forest_type", TYPES),
    "population_vs_report": {"acres": float(pop_tab.iloc[-1]["acres"]), "report": sum(report.values()), "diff_pct": float(pop_tab.iloc[-1]["diff_pct"]), "wilderness_excluded": EXCL_WILD},
    "empty_vp9_classes_at_min": {"count": empty_vp9},
    "empty_vp10_classes_at_min": {"count": empty_vp10},
    "half_a_caty_off_by_more_than_1": {"count": int((caty_chk["diff"].abs() > 1).sum())},
    "covariate_flags": {"count": len(flag)},
    "cells_collapsed": {"count": int(len(collapse_log))},
    "spatial_balance": sb,
}
for kk, v in checks.items():
    log.info(f"QA {kk}: {v}")
pd.json_normalize(checks).T.to_csv(O / "qa_report.csv", header=False)
log.info("evaluation complete; read outputs/qa_report.csv before the freeze")

## 7. Export

Two GeoPackages in EPSG:26910: `plots_design_<date>.gpkg`, the public layer with `run.publish_strip_fields` removed, and `plots_design_<date>_internal.gpkg` with everything. Primaries and legacy sites carry the installation order and nested level; backups carry their GRTS rank within half and stratum as the backup order. Every site is tagged with the 400 ha owl cell key and the 133 ha sample design cell key from the lattice `06_tessellation` fitted (`outputs/tessellation_lattice_parameters.csv`); without that file a placeholder lattice is used and the `hex_source` field says so. The installation order and the backup list also go out as CSVs for the RFP appendix.

**Backup rule (Andy decides, with Mason for the RFP text).** A backup replaces a primary only with TRPA's written concurrence for access denial, safety, or site not as mapped, and the replacement is recorded with the reason. Backups are taken in order within the same half and stratum; a design coordinate is never moved.

Decided: the two layers, stripped fields, hex tagging, the CRS. Andy decides: the backup list rule and how many backups per stratum go into the RFP. Module: `src/design_template.hex_keys`; `src/tessellation`.

In [ ]:
owl_key, fine_key, hex_src = dt.hex_keys(sel["x"], sel["y"], cfg, O / "tessellation_lattice_parameters.csv", log)
sel["owl_cell_key"], sel["fine_cell_key"], sel["hex_source"] = owl_key, fine_key, hex_src
sel["backup_order"] = np.where(sel["status"] == "backup", sel["grts_rank"], np.nan)
sel["selection"] = SELECTION
sel["design_crs"] = WCRS
sel["strata_source"] = cfg["strata"]["source"]
sel["design_date"] = STAMP

drop_internal = ["tpi", "inclusion_weight"]
public_cols = [c for c in sel.columns if c not in set(cfg["run"]["publish_strip_fields"]) | set(drop_internal)]
internal = gpd.GeoDataFrame(sel, geometry=gpd.points_from_xy(sel["x"], sel["y"]), crs=WCRS)
for c in internal.columns:
    if internal[c].dtype == object and c != "geometry":
        internal[c] = internal[c].astype(str).where(internal[c].notna(), None)
internal.to_file(O / f"plots_design_{STAMP}_internal.gpkg", driver="GPKG")
internal[public_cols + ["geometry"]].to_file(O / f"plots_design_{STAMP}.gpkg", driver="GPKG")

order_cols = [c for c in ["install_order", "plot_id", "half", "stratum", "cell_id", "forest_type", "level", "status", "legacy_source",
                          "access_class", "state", "owner", "owl_cell_key", "fine_cell_key", "x", "y"] if c in sel]
prim = sel[sel["status"].isin(["primary", "legacy"])].sort_values("install_order").copy()
prim["install_order"] = prim["install_order"].astype(int)
prim[order_cols].to_csv(O / f"installation_order_{STAMP}.csv", index=False)
back = sel[sel["status"] == "backup"].sort_values(["half", "stratum", "backup_order"])
back[[c for c in ["plot_id", "half", "stratum", "cell_id", "forest_type", "backup_order", "access_class", "state", "owl_cell_key", "fine_cell_key", "x", "y"] if c in back]].to_csv(O / f"backup_list_{STAMP}.csv", index=False)
log.info(f"wrote plots_design_{STAMP}.gpkg ({len(prim)} primaries and legacy, {len(back)} backups), _internal, installation_order, backup_list; hex keys from {hex_src}")
print(prim[order_cols].head(12).to_string(index=False))
if FREEZE:
    log.info(f"FROZEN design: seed {cfg['draw']['seed']}, strata.source {cfg['strata']['source']}, selection {SELECTION}, {STAMP}. Nothing after this changes without restarting review.")